# JuanSign — Google Colab Training Notebook

**Before running:** Go to `Runtime > Change runtime type > T4 GPU`

### Steps
1. Check GPU
2. Mount Google Drive
3. Upload repo zip from Drive
4. Install packages
5. (Optional) Run preprocessing
6. Train model
7. Download trained weights

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("NO GPU FOUND")
    print("Go to: Runtime > Change runtime type > Hardware accelerator > T4 GPU")

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────────
# You need to upload the repo zip to your Google Drive first.
# Zip the entire 'Thesis' folder on your PC, then upload to Drive root.

from google.colab import drive
drive.mount('/content/drive')

import os
# List Drive root to confirm your zip is there
print(os.listdir('/content/drive/MyDrive/'))

In [ ]:
# ── Cell 3: Extract repo from Drive ──────────────────────────────────────────
# EDIT this to match your zip filename in Google Drive
ZIP_NAME = 'JuanSign_Thesis.zip'   # <-- change this to your actual zip name

import shutil, os

src = f'/content/drive/MyDrive/{ZIP_NAME}'
dst = f'/content/{ZIP_NAME}'

print(f'Copying {ZIP_NAME} from Drive...')
shutil.copy(src, dst)

print('Extracting...')
!unzip -q /content/{ZIP_NAME} -d /content/juansign

# List what was extracted
print(os.listdir('/content/juansign'))

In [ ]:
# ── Cell 3b (ALTERNATIVE): Clone from GitHub instead of Drive ─────────────────
# Only use this cell if your repo is PUBLIC on GitHub.
# Comment out Cell 3 above if using this one.

GITHUB_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # <-- fill in

!git clone {GITHUB_URL} /content/juansign
import os
print(os.listdir('/content/juansign'))

In [ ]:
# ── Cell 4: Set working directory ────────────────────────────────────────────
import os

# Find the repo root (the folder that contains 'ml-model')
# Adjust path if your zip extracted to a nested folder
candidates = ['/content/juansign', '/content/juansign/Thesis']
REPO_ROOT = None
for c in candidates:
    if os.path.isdir(os.path.join(c, 'ml-model')):
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    # Manual fallback — list the extracted folder so you can find it
    for root, dirs, files in os.walk('/content/juansign'):
        if 'ml-model' in dirs:
            REPO_ROOT = root
            break

os.chdir(REPO_ROOT)
print(f'Working directory: {os.getcwd()}')
print(os.listdir('.'))

In [ ]:
# ── Cell 5: Install packages ─────────────────────────────────────────────────
# Colab already has torch, torchvision, numpy, matplotlib, scikit-learn.
# We only need to install mediapipe and seaborn.

!pip install -q mediapipe==0.10.11 seaborn==0.13.2 tensorboard==2.16.2

# Verify
import torch, cv2, mediapipe, seaborn
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
print(f'OpenCV   : {cv2.__version__}')
print(f'MediaPipe: {mediapipe.__version__}')

## Data Preprocessing (only if you have raw videos)
Skip Cells 6 & 7 if your `processed_output/frame_extracted/` folder is already in the zip.

In [ ]:
# ── Cell 6: Split raw videos into train/test/val ──────────────────────────────
# Source: processed_output/raw_data/<letter>/*.mp4
# Output: unprocessed_input/{training,testing,validation}_data/<letter>/

!python ml-model/src/data_splitter.py

In [ ]:
# ── Cell 7: Extract 16 frames per video clip ──────────────────────────────────
# Input : unprocessed_input/<split>/<letter>/*.mp4
# Output: processed_output/frame_extracted/<split>/<letter>/clipXXX/frame####.jpg
# NOTE  : This auto-downloads hand_landmarker.task (~30 MB) on first run.
#         Resume-safe: skips clips already extracted.

!python ml-model/src/frame_extractor.py

## Training

In [ ]:
# ── Cell 8: Verify dataset is in place ───────────────────────────────────────
import os

BASE = 'ml-model/processed_output/frame_extracted'
for split in ['training_data', 'testing_data', 'validation_data']:
    split_path = os.path.join(BASE, split)
    if os.path.isdir(split_path):
        letters = os.listdir(split_path)
        total_clips = sum(
            len(os.listdir(os.path.join(split_path, l)))
            for l in letters
            if os.path.isdir(os.path.join(split_path, l))
        )
        print(f'{split}: letters={letters}, clips={total_clips}')
    else:
        print(f'MISSING: {split_path}')

In [ ]:
# ── Cell 9: Train the model ───────────────────────────────────────────────────
# Trains for up to 25 epochs with early stopping (patience=5).
# Best checkpoint saved to: ml-model/juansignmodel/juansign_model.pth

!python ml-model/src/train.py

In [ ]:
# ── Cell 10: Launch TensorBoard ──────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir ml-model/runs

In [ ]:
# ── Cell 11: Download trained weights ────────────────────────────────────────
from google.colab import files
import os

weights_path = 'ml-model/juansignmodel/juansign_model.pth'
if os.path.exists(weights_path):
    files.download(weights_path)
    print('Download started.')
else:
    print('Weights file not found — training may not have finished.')

In [ ]:
# ── Cell 12 (OPTIONAL): Save weights back to Google Drive ────────────────────
import shutil

dst = '/content/drive/MyDrive/juansign_model.pth'
shutil.copy('ml-model/juansignmodel/juansign_model.pth', dst)
print(f'Saved to Drive: {dst}')